# M5.0 · 负结果：CPU 卸载并不免费

**核心问题：** 有 CPU/GPU 重叠就一定提高吞吐吗？根据 H2 原始记录重算负结果：虽然存在约50秒的真实重叠，低效的 PyTorch CPU 编码仍拖垮整体流水线。默认不重跑高成本实验。

**学习闭环：** 先预测 → 检查固定证据 → 独立计算 → 批判性解释 → 设计受控实验。

对应理论课：[打开 course/M05_parallel_heterogeneous/LESSON.zh-CN.md](../course/M05_parallel_heterogeneous/LESSON.zh-CN.md)。切换语言：[English](./05_cpu_offload_negative.ipynb)。

**默认模式：** 纯 CPU、无需联网、只读仓库证据。不会自动启动 GPU。


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "lab/reference_runs.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the repository root (see notebooks/README).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from lab import notebook_support as ns
import statistics
print("Course repository:", ROOT)
print("Mode: OFFLINE / committed reference evidence; NO GPU work is launched by default.")


## 检查真实数据并独立计算

按照顺序执行；每个比例都要解释分母与适用范围。

In [ ]:
h2 = ns.read_json("evidence/results/p5-H2-H2-pipeline/H2_RESULTS.json")
a = h2["A_sequential"]["total_s"]
b = h2["B_pipelined"]["total_s"]
overlap = h2["B_pipelined"]["overlap_s_total"]
print(f"Four requests: GPU-native sequential {a:.3f}s, CPU+GPU naive pipeline {b:.3f}s")
print(f"Real overlapping work {overlap:.3f}s, relative throughput A/B={a/b:.3f}x")
print(f"Pipeline is {b/a:.2f}x SLOWER (same four images), despite proven overlap.")
assert h2["B_pipelined"]["overlap_proven"] is True
plot_values = {"GPU native": a, "PyTorch CPU offload": b}
plot_ylabel, plot_title = "4-request wall seconds", "H2 negative result (NOT a new run)"


In [ ]:
SHOW_PLOT = False  # Optional: requires matplotlib in the notebook kernel
if SHOW_PLOT:
    import matplotlib.pyplot as plt
    labels = list(plot_values)
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.bar(labels, [plot_values[label] for label in labels])
    ax.set_ylabel(plot_ylabel)
    ax.set_title(plot_title)
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()
else:
    print("Optional plot skipped; set SHOW_PLOT=True to visualize the measured values.")


In [ ]:
# A genuine algorithm exercise: simulate a two-stage CPU->GPU pipeline.
# This is a toy analytical scheduler, NOT a measured H2 timing reconstruction.
def two_stage_makespan(cpu_s, gpu_s):
    cpu_available = gpu_available = 0.0
    for cpu, gpu in zip(cpu_s, gpu_s):
        cpu_available += cpu
        gpu_available = max(cpu_available, gpu_available) + gpu
    return gpu_available

gpu_times = [12.0] * 4
for encoder in (150.0, 0.5):
    toy_wall = two_stage_makespan([encoder] * 4, gpu_times)
    print(f"Toy scenario CPU encode={encoder:.1f}s x4: pipeline wall={toy_wall:.1f}s")
print("H2's real 737s includes resource contention, setup and handoff; use H2 JSON for claims.")


## 学生思考 / 自检

画出关键路径，指出吞吐受哪个执行引擎限制。设计一个可以挽救负结果的 CPU 文本编码器优化假设，并说明如何验证。

**证据规范：** 用自己的语言完成回答。本 Notebook 重算的是已提交记录，不是你的 GPU 实测。如果自愿运行 smoke，结果必须放在 `runs/`，不能宣传为正式全量统计。
